# 11 – Business analytics with Metabase

**Goal (Task 5, part 4):** set up **Metabase**, connect it to **Trino**, and build a **dashboard with three charts on the `gold` zone** that answer business questions about PawHome.

This notebook does the setup through **Metabase's API** (repeatable, nothing to click):
1. creates the Metabase **admin account** (test login of your local Metabase, from `docker-compose.yml`): `admin@pawhome.local` / `PawHome-metabase-2026`
2. adds the **Trino connection** (driver "Starburst", which also works for Trino): host `trino-coordinator`, port `8080`, catalog `pile`, schema `gold`
3. creates **3 questions** (SQL on `pile.gold`) and a **dashboard** with them

Open Metabase afterwards: **http://localhost:3001** (port 3001, because 3000 is the PawHome backend).

| # | Business question | Gold table | Chart |
|---|---|---|---|
| 1 | How is adoption demand developing month by month? | `shelter_monthly_activity` | line |
| 2 | Which shelters place animals most successfully (last 12 months)? | `shelter_monthly_activity` | horizontal bars |
| 3 | Which species find a home most easily? | `breed_popularity` | bars |

In [ ]:
import os, time, requests
import pandas as pd
from trino_pile import sql

MB = os.getenv("METABASE_URL", "http://metabase:3000")
EMAIL = os.getenv("MB_ADMIN_EMAIL", "admin@pawhome.local")
PASSWORD = os.getenv("MB_ADMIN_PASSWORD", "PawHome-metabase-2026")
session = requests.Session()


def api(method, path, **kwargs):
    """Call the Metabase API; show Metabase's own error message if something goes wrong."""
    response = session.request(method, f"{MB}{path}", timeout=120, **kwargs)
    if response.status_code >= 400:
        raise RuntimeError(f"{method} {path} -> {response.status_code}: {response.text[:800]}")
    return response.json() if response.text else None


# Metabase needs 1-2 minutes to start the first time
for _ in range(60):
    try:
        if requests.get(f"{MB}/api/health", timeout=5).json().get("status") == "ok":
            break
    except Exception:
        pass
    time.sleep(5)
print("Metabase is up:", MB)

## Step 1 – Admin account (first start) or log in

In [ ]:
properties = api("GET", "/api/session/properties")
if not properties.get("has-user-setup"):
    result = api("POST", "/api/setup", json={
        "token": properties["setup-token"],
        "user": {"first_name": "PawHome", "last_name": "Admin", "email": EMAIL,
                 "password": PASSWORD, "site_name": "PawHome"},
        "prefs": {"site_name": "PawHome", "site_locale": "en", "allow_tracking": False},
    })
    print("Created the Metabase admin account:", EMAIL)
else:
    result = api("POST", "/api/session", json={"username": EMAIL, "password": PASSWORD})
    print("Logged in as", EMAIL)
session.headers["X-Metabase-Session"] = result["id"]

## Step 2 – Connect Metabase to Trino

In [ ]:
DB_NAME = "PawHome pile (Trino)"
databases = api("GET", "/api/database")
databases = databases["data"] if isinstance(databases, dict) else databases
existing = [d for d in databases if d["name"] == DB_NAME]

if existing:
    db_id = existing[0]["id"]
    print("Trino connection already exists, id", db_id)
else:
    db_id = api("POST", "/api/database", json={
        "engine": "starburst",              # Metabase's Starburst driver = Trino
        "name": DB_NAME,
        "details": {"host": "trino-coordinator", "port": 8080, "catalog": "pile",
                    "schema": "gold", "user": "metabase", "ssl": False},
    })["id"]
    print("Added Trino connection, id", db_id)
api("POST", f"/api/database/{db_id}/sync_schema")   # let Metabase discover the gold tables

## Step 3 – Three questions on the gold zone

Each question is a SQL query that **Trino** runs on the **gold** Iceberg tables. Below, the same SQL is also run directly through Trino so you can see the numbers.

In [ ]:
QUESTIONS = [
    {
        "name": "1. Adoption demand per month",
        "description": "How is adoption demand developing month by month? Applications, successful applications and adoptions over the last 2 years.",
        "display": "line",
        "sql": """
SELECT month_start AS month,
       sum(applications) AS applications,
       sum(successful)   AS successful,
       sum(adoptions)    AS adoptions
FROM pile.gold.shelter_monthly_activity
WHERE month_start >= date_add('month', -23, date_trunc('month', current_date))
GROUP BY month_start
ORDER BY month_start""",
        "viz": {"graph.dimensions": ["month"], "graph.metrics": ["applications", "successful", "adoptions"],
                "graph.x_axis.title_text": "Month", "graph.y_axis.title_text": "Number"},
    },
    {
        "name": "2. Most successful shelters (last 12 months)",
        "description": "Which shelters place animals most successfully? Share of applications that were approved or ended in adoption.",
        "display": "row",
        "sql": """
SELECT shelter_name,
       round(1.0 * sum(successful) / sum(applications), 3) AS success_rate,
       sum(applications) AS applications
FROM pile.gold.shelter_monthly_activity
WHERE month_start >= date_add('month', -12, date_trunc('month', current_date))
GROUP BY shelter_name
HAVING sum(applications) >= 10
ORDER BY success_rate DESC
LIMIT 10""",
        "viz": {"graph.dimensions": ["shelter_name"], "graph.metrics": ["success_rate"],
                "graph.y_axis.title_text": "Success rate"},
    },
    {
        "name": "3. Chance of finding a home by species",
        "description": "Which species find a home most easily, and how long do they stay in the shelter?",
        "display": "bar",
        "sql": """
SELECT species,
       round(1.0 * sum(found_home) / sum(animals), 3)              AS found_home_rate,
       sum(animals)                                                AS animals,
       round(1.0 * sum(avg_days_in_shelter * animals) / sum(animals)) AS avg_days_in_shelter
FROM pile.gold.breed_popularity
GROUP BY species
ORDER BY found_home_rate DESC""",
        "viz": {"graph.dimensions": ["species"], "graph.metrics": ["found_home_rate"],
                "graph.y_axis.title_text": "Share that found a home"},
    },
]

for q in QUESTIONS:
    print(q["name"])
    display(sql(q["sql"]).head(12))

In [ ]:
existing_cards = {c["name"]: c["id"] for c in api("GET", "/api/card")}
card_ids = []
for q in QUESTIONS:
    body = {
        "name": q["name"], "description": q["description"], "display": q["display"],
        "visualization_settings": q["viz"],
        "dataset_query": {"type": "native", "database": db_id, "native": {"query": q["sql"]}},
    }
    if q["name"] in existing_cards:
        card = api("PUT", f"/api/card/{existing_cards[q['name']]}", json=body)
    else:
        card = api("POST", "/api/card", json=body)
    card_ids.append(card["id"])
    rows = api("POST", f"/api/card/{card['id']}/query")["data"]["rows"]
    print(f"card {card['id']}: {q['name']} -> {len(rows)} rows from Trino")

## Step 4 – The dashboard

In [ ]:
DASHBOARD = "PawHome – adoption analytics (gold zone)"
found = api("GET", "/api/search", params={"q": DASHBOARD, "models": "dashboard"})
found = [d for d in (found["data"] if isinstance(found, dict) else found) if d["name"] == DASHBOARD]

if found:
    dashboard_id = found[0]["id"]
else:
    dashboard_id = api("POST", "/api/dashboard", json={
        "name": DASHBOARD,
        "description": "Three business questions answered from the gold zone of the PawHome data pile, queried through Trino.",
    })["id"]

layout = [  # Metabase grid is 24 columns wide
    {"id": -1, "card_id": card_ids[0], "row": 0, "col": 0,  "size_x": 24, "size_y": 7},
    {"id": -2, "card_id": card_ids[1], "row": 7, "col": 0,  "size_x": 12, "size_y": 8},
    {"id": -3, "card_id": card_ids[2], "row": 7, "col": 12, "size_x": 12, "size_y": 8},
]
api("PUT", f"/api/dashboard/{dashboard_id}", json={"dashcards": layout})
print("Dashboard ready: http://localhost:3001/dashboard/%s" % dashboard_id)
print("Log in with", EMAIL, "/", PASSWORD)